# Phase 1 — Data Acquisition

Reproducible pipeline for the four data streams the project depends on:

1. **NCES Digest** — high school graduate counts and enrollment projections (tables 219.10, 303.10, 302.10).
2. **IPEDS** — Admissions, Institutional Characteristics, and Fall Enrollment by race/ethnicity (ADM, IC, EFFY).
3. **Synthetic applicants** — 50,000 inquiry-level rows calibrated to published IPEDS yield rates.
4. **ASU corpus** — public scrape of `admission.asu.edu` + `students.asu.edu` for the chatbot knowledge base.

Run all cells from a clean checkout to reproduce. All downloads are idempotent — existing files are skipped.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import logging
import pandas as pd

from src.utils import set_seeds, NCES_DIR, IPEDS_DIR, SYNTHETIC_DIR, CORPUS_DIR

logging.basicConfig(level=logging.INFO, format='%(message)s')
set_seeds()

## 1. NCES Digest tables

Three tables drive the demographic forecasting module:

| Table | Topic | Source year |
|---|---|---|
| 219.10 | HS graduates by state | Digest 2022 |
| 303.10 | Total fall enrollment in degree-granting institutions | Digest 2021 (latest published) |
| 302.10 | Recent HS completers enrollment, by sex/race | Digest 2022 |

In [ ]:
from src.data_acquisition import fetch_nces_tables

nces_results = fetch_nces_tables()
nces_results

In [ ]:
# Spot-check: read table 219.10 (HS graduates by state) — header rows vary, so peek raw
raw_219 = pd.read_excel(NCES_DIR / 'tabn219.10.xls', header=None)
print('shape:', raw_219.shape)
raw_219.head(8)

## 2. IPEDS surveys

Three surveys for fiscal years 2018–2022:

| Survey | Topic |
|---|---|
| ADM | Admissions: applications, admits, enrollees, yield by institution |
| IC | Institutional Characteristics: control, level, region |
| EFFY | 12-month enrollment by race/ethnicity |

EF subparts (A=residence, B=age, C=race, D=full/part-time) are not fetched here — EFFY covers our race-ethnicity needs. Adjust `IPEDS_FILES_TEMPLATE` if subparts are needed later.

In [ ]:
from src.data_acquisition import fetch_ipeds

ipeds_results = fetch_ipeds()
print(f'fetched {sum(ipeds_results.values())}/{len(ipeds_results)} IPEDS zips')

In [ ]:
# Spot-check: peek inside ADM2022.zip
import zipfile

with zipfile.ZipFile(IPEDS_DIR / 'ADM2022.zip') as zf:
    print('files in ADM2022.zip:', zf.namelist())
    member = zf.namelist()[0]
    adm = pd.read_csv(zf.open(member), encoding='latin-1', low_memory=False)

print('shape:', adm.shape)
adm.head(3)

## 3. Synthetic applicants

Generate 50,000 inquiry-level rows. Calibrated yield targets per institution segment (within ±2pp):

| Segment | Target yield |
|---|---|
| R1 | 33% |
| Regional state | 22% |
| Private LAC | 28% |
| Community college | 50% |
| Online | 40% |

A small `first_gen` penalty is intentionally baked into the lead logit so the bias audit in Phase 3 has something real to detect and mitigate.

In [ ]:
from src.data_synthesis import generate_applicants, yield_rate_by_segment

df = generate_applicants(n=50_000, seed=42)
out_path = SYNTHETIC_DIR / 'applicants.csv'
df.to_csv(out_path, index=False)
print(f'wrote {len(df):,} rows -> {out_path.relative_to(Path.cwd().parent)}')
df.head(3)

In [ ]:
# EDA — calibration check
yield_rate_by_segment(df).round(4)

In [ ]:
# Funnel summary
n = len(df)
applied = int(df['inquired_to_applied'].sum())
admitted = int(df['admitted'].sum())
enrolled = int(df['admit_to_enroll'].sum())

pd.DataFrame({
    'stage': ['inquired', 'applied', 'admitted', 'enrolled'],
    'count': [n, applied, admitted, enrolled],
    'rate_from_prev': [1.0, applied/n, admitted/applied, enrolled/admitted],
}).round(3)

In [ ]:
# Distribution sanity
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
df['hs_gpa'].hist(ax=axes[0], bins=40); axes[0].set_title('HS GPA')
df['sat_score'].hist(ax=axes[1], bins=40); axes[1].set_title('SAT')
df['distance_miles'].clip(0, 1500).hist(ax=axes[2], bins=40); axes[2].set_title('Distance (clipped at 1500)')
plt.tight_layout()

In [ ]:
# Leakage check — confirm targets are nested correctly
assert df.query('inquired_to_applied == 0')['admitted'].sum() == 0, 'admit without apply'
assert df.query('admitted == 0')['admit_to_enroll'].sum() == 0, 'enroll without admit'
print('leakage check passed')

## 4. ASU corpus scrape

Public crawl of admission/students/tuition ASU subdomains. Robots.txt is respected (custom Disallow-only parser — Python's stdlib `urllib.robotparser` mishandles `Allow` directives with `$` anchors). 1-req/sec throttle.

Re-run is idempotent only at the file level — existing markdown files are not regenerated unless deleted. Default cap: 250 pages.

In [ ]:
existing = list(CORPUS_DIR.glob('*.md'))
print(f'existing corpus files: {len(existing)}')

if len(existing) < 200:
    from src.data_acquisition import scrape_asu_corpus, ASU_SEED_URLS
    n_written = scrape_asu_corpus(ASU_SEED_URLS, max_pages=250, throttle_seconds=0.7)
    print(f'wrote {n_written} new files')
else:
    print('skip — corpus already populated')

In [ ]:
# Quick chunk-size preview — confirm files are non-trivial
sizes = [p.stat().st_size for p in CORPUS_DIR.glob('*.md')]
if sizes:
    s = pd.Series(sizes)
    print(f'corpus files: {len(s):,}')
    print(f'total size: {s.sum() / 1024:.1f} KB')
    print(f'mean / median bytes: {s.mean():.0f} / {s.median():.0f}')

## 5. Phase 1 acceptance gates

- All NCES tables present in `data/raw/nces/`
- All IPEDS zips present in `data/raw/ipeds/`
- `data/synthetic/applicants.csv` has 50,000 rows, yield rates within ±2pp of targets
- `data/corpus/` has ≥200 markdown files
- See `docs/data_dictionary.md` for full column reference

In [ ]:
from src.utils import PROJECT_ROOT

checks = {
    'NCES tables (219.10, 302.10, 303.10)': all((NCES_DIR / f).exists() for f in ['tabn219.10.xls', 'tabn302.10.xls', 'tabn303.10.xls']),
    'IPEDS zips ≥ 12': len(list(IPEDS_DIR.glob('*.zip'))) >= 12,
    'synthetic applicants exists': (SYNTHETIC_DIR / 'applicants.csv').exists(),
    'synthetic applicants 50k rows': len(pd.read_csv(SYNTHETIC_DIR / 'applicants.csv')) == 50_000,
    'corpus ≥ 200 files': len(list(CORPUS_DIR.glob('*.md'))) >= 200,
}
for k, v in checks.items():
    print(('OK ' if v else 'FAIL '), k)
assert all(checks.values()), 'Phase 1 acceptance failed'